# **CSCE 5218 / CSCE 4930 Deep Learning**

# **The Perceptron** (20 pt)


In [6]:
# Take a peek at the datasets
!head train.dat
!head test.dat

A1	A2	A3	A4	A5	A6	A7	A8	A9	A10	A11	A12	A13	
1	1	0	0	0	0	0	0	1	1	0	0	1	0
0	0	1	1	0	1	1	0	0	0	0	0	1	0
0	1	0	1	1	0	1	0	1	1	1	0	1	1
0	0	1	0	0	1	0	1	0	1	1	1	1	0
0	1	0	0	0	0	0	1	1	1	1	1	1	0
0	1	1	1	0	0	0	1	0	1	1	0	1	1
0	1	1	0	0	0	1	0	0	0	0	0	1	0
0	0	0	1	1	0	1	1	1	0	0	0	1	0
0	0	0	0	0	0	1	0	1	0	1	0	1	0
A1	A2	A3	A4	A5	A6	A7	A8	A9	A10	A11	A12	A13
1	1	1	1	0	0	1	1	0	0	0	1	1	0
0	0	0	1	0	0	1	1	0	1	0	0	1	0
0	1	1	1	0	1	1	1	1	0	0	0	1	0
0	1	1	0	1	0	1	1	1	0	1	0	1	0
0	1	0	0	0	1	0	1	0	1	0	0	1	0
0	1	1	0	0	1	1	1	1	1	1	0	1	0
0	1	1	1	0	0	1	1	0	0	0	1	1	0
0	1	0	0	1	0	0	1	1	0	1	1	1	0
1	1	1	1	0	0	1	1	0	0	0	0	1	0


### Build the Perceptron Model

You will need to complete some of the function definitions below.  DO NOT import any other libraries to complete this.

In [7]:
import math
import itertools
import re


# Corpus reader, all columns but the last one are coordinates;
#   the last column is the label
def read_data(file_name):
    f = open(file_name, 'r')

    data = []
    # Discard header line
    f.readline()
    for instance in f.readlines():
        if not re.search('\t', instance): continue
        instance = list(map(int, instance.strip().split('\t')))
        # Add a dummy input so that w0 becomes the bias
        instance = [-1] + instance
        data += [instance]
    return data


def dot_product(array1, array2):
    return sum(array1[i] * array2[i] for i in range(len(array1)))

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def output(weight, instance):
    return sigmoid(dot_product(weight, instance))

def predict(weights, instance):
    return 1 if output(weights, instance) >= 0.5 else 0

# Accuracy = percent of correct predictions
def get_accuracy(weights, instances):
    # You do not to write code like this, but get used to it
    correct = sum([1 if predict(weights, instance) == instance[-1] else 0
                   for instance in instances])
    return correct * 100 / len(instances)


# Train a perceptron with instances and hyperparameters:
#       lr (learning rate)
#       epochs
# The implementation comes from the definition of the perceptron
#
# Training consists on fitting the parameters which are the weights
# that's the only thing training is responsible to fit
# (recall that w0 is the bias, and w1..wn are the weights for each coordinate)
#
# Hyperparameters (lr and epochs) are given to the training algorithm
# We are updating weights in the opposite direction of the gradient of the error,
# so with a "decent" lr we are guaranteed to reduce the error after each iteration.
def train_perceptron(instances, lr, epochs):

    #TODO: name this step
    weights = [0] * (len(instances[0])-1)

    for _ in range(epochs):
        for instance in instances:
            #TODO: name these steps
            in_value = dot_product(weights, instance)
            output = sigmoid(in_value)
            error = instance[-1] - output
            #TODO: name these steps
            for i in range(0, len(weights)):
                weights[i] += lr * error * output * (1-output) * instance[i]

    return weights

## Run it

In [8]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")

lr = 0.005
epochs = 5

weights = train_perceptron(instances_tr, lr, epochs)
accuracy = get_accuracy(weights, instances_te)

print(f"#tr: {len(instances_tr):3}, epochs: {epochs:3}, learning rate: {lr:.3f}; "
      f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

#tr: 400, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0


## Questions

Answer the following questions. Include your implementation and the output for each question.



### Question 1

In `train_perceptron(instances, lr, epochs)`, we have the follosing code:
```
in_value = dot_product(weights, instance)
output = sigmoid(in_value)
error = instance[-1] - output
```

Why don't we have the following code snippet instead?
```
output = predict(weights, instance)
error = instance[-1] - output
```

#### TODO Add your answer here:


We do not use predict(weights, instance) during training because predict() returns a binary output (0 or 1). Training requires the continuous probability value produced by the sigmoid function in order to compute gradients properly.

If we used predict(), the gradient term would disappear because binary values do not allow smooth updates. The perceptron with sigmoid uses gradient descent, which requires the differentiable sigmoid output. Therefore, we must use:
output = sigmoid(in_value)
error = instance[-1] - output




### Question 2
Train the perceptron with the following hyperparameters and calculate the accuracy with the test dataset.

```
tr_percent = [5, 10, 25, 50, 75, 100] # percent of the training dataset to train with
num_epochs = [5, 10, 20, 50, 100]              # number of epochs
lr = [0.005, 0.01, 0.05]              # learning rate
```

TODO: Write your code below and include the output at the end of each training loop (NOT AFTER EACH EPOCH)
of your code.The output should look like the following:
```
# tr:  20, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
[and so on for all the combinations]
```
You will get different results with different hyperparameters.

#### TODO Add your answer here (code and output in the format above)


In [9]:
instances_tr = read_data("train.dat")
instances_te = read_data("test.dat")

tr_percent = [5, 10, 25, 50, 75, 100]
num_epochs = [5, 10, 20, 50, 100]
lr_array = [0.005, 0.01, 0.05]

for lr in lr_array:
    for tr_size in tr_percent:
        size = round(len(instances_tr) * tr_size / 100)
        pre_instances = instances_tr[0:size]

        for epochs in num_epochs:
            weights = train_perceptron(pre_instances, lr, epochs)
            accuracy = get_accuracy(weights, instances_te)

            print(f"# tr: {len(pre_instances):3}, epochs: {epochs:3}, "
                  f"learning rate: {lr:.3f}; "
                  f"Accuracy (test, {len(instances_te)} instances): {accuracy:.1f}")

# tr:  20, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs:  50, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  20, epochs: 100, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  40, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  40, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  40, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  40, epochs:  50, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr:  40, epochs: 100, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr: 100, epochs:   5, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr: 100, epochs:  10, learning rate: 0.005; Accuracy (test, 100 instances): 68.0
# tr

### Question 3
Write a couple paragraphs interpreting the results with all the combinations of hyperparameters. Drawing a plot will probably help you make a point. In particular, answer the following:
- A. Do you need to train with all the training dataset to get the highest accuracy with the test dataset?
- B. How do you justify that training the second run obtains worse accuracy than the first one (despite the second one uses more training data)?

#tr: 100, epochs:  20, learning rate: 0.050; Accuracy (test, 100 instances): 71.0
#tr: 200, epochs:  20, learning rate: 0.005; Accuracy (test, 100 instances): 68.0

- C. Can you get higher accuracy with additional hyperparameters (higher than `80.0`)?
- D. Is it always worth training for more epochs (while keeping all other hyperparameters fixed)?

#### TODO: Add your answer here :


### A. Do you need to train with all the training dataset to get the highest accuracy?
No. The highest test accuracy does not always require using 100% of the training data. In several runs, smaller subsets achieved similar or even better accuracy. This suggests that additional data may introduce noise or redundancy, and that model performance depends more on the quality of data and hyperparameters than simply quantity

### B. Why can more data give worse accuracy?

Training with more data can sometimes reduce accuracy due to:

Noise in additional training samples

Suboptimal learning rate

Insufficient epochs for convergence

Overfitting to complex patterns

For example:
#tr: 100, epochs: 20, lr: 0.050 → 71.0%
#tr: 200, epochs: 20, lr: 0.005 → 68.0%
The second run used more data but a smaller learning rate, which may have slowed convergence and resulted in lower performance.

### C. Can you get higher than 80%?

Yes. By increasing epochs and tuning learning rate (e.g., lr=0.01 or 0.05 and epochs=100+), it is possible to reach or exceed 80% accuracy depending on the dataset behavior.

### D. Is more epochs always better?

No. More epochs can improve learning initially, but excessive training may cause:

Overfitting

Minimal accuracy improvement

Increased computation time

There is usually an optimal number of epochs beyond which improvement plateaus.